In [1]:
import pandas as pd
reference_submission = pd.read_csv('reference_submission.csv')
test_weights = pd.read_csv('test_weights.csv')      
# Extract unique_id from id in reference_submission
reference_submission['unique_id'] = reference_submission['id'].str.split('_').str[0].astype(int)

# Merge with test_weights on unique_id
merged_df = reference_submission.merge(test_weights, on='unique_id', how='left')

#check merge is correct, one weight for each unique_id
#filtered_df = merged_df[merged_df['unique_id'] == 2385]
#print(filtered_df)

# Drop unique_id if not needed
df = merged_df.drop(columns=['unique_id'])

print(df)

                    id   sales_hat    weight
0      2385_2024-06-03   65.990798  4.024659
1      3148_2024-06-03   32.579229  2.073344
2      4580_2024-06-03    8.816822  2.293700
3       745_2024-06-03   53.803901  1.173380
4       549_2024-06-03   67.328265  1.420731
...                ...         ...       ...
47016  4123_2024-06-16   13.982063  3.130117
47017  5040_2024-06-16   14.201070  1.472375
47018  5188_2024-06-16   28.548889  1.964792
47019  2746_2024-06-16   47.308901  5.078191
47020  2304_2024-06-16  102.691358  2.623342

[47021 rows x 3 columns]


In [3]:
import numpy as np
import pandas as pd

# Use merged_df as the base data
df = merged_df.copy()

# Given information
current_mae = 19.02669
target_mae = 30

# Compute the required additional MAE
required_additional_mae = target_mae - current_mae
 
# Generate probability weights ∝ 1 / (weight³), normalized
df['weight_factor'] = 1 / (df['weight']**3 + 1e-6)  # Avoid division by zero
df['weight_factor'] /= df['weight_factor'].sum()  # Normalize so it sums to 1

# Generate skewed noise using Laplace distribution (sharp peak, heavy tails)
np.random.seed(42)  # For reproducibility
raw_noise = np.random.laplace(0, 1, size=len(df))  # Base noise

# Scale the noise so the total MAE equals target_mae
scaled_noise = raw_noise * df['weight_factor'] * required_additional_mae * len(df)

# Apply the noise to sales_hat
df['sales_hat'] = df['sales_hat'] + scaled_noise

# Verify that the final MAE is approximately 30
estimated_mae = np.mean(np.abs(scaled_noise)) + current_mae
print(f"Estimated Final MAE: {estimated_mae:.4f}")  # Should be ~30

# Drop unnecessary columns
df = df.drop(columns=['weight_factor', 'weight'], errors='ignore')

# Save the modified dataset
df.to_csv("modified_predictions_skewed.csv", index=False)

print("Modified dataset saved as 'modified_predictions_skewed.csv'.")


Estimated Final MAE: 29.4903
Modified dataset saved as 'modified_predictions_skewed.csv'.
